# UrbanEV Shenzhen benchmark | Issue #96

This notebook extends Đức's independently implemented pilot to four stations and the three stochastic seeds in Issue #96. It records 160 non-spatial and 12 spatial results when all jobs complete. The 14-day training prefix, 90-day replay, 24-hour history and 24-hour forecast horizon are fixed.

The run uses Cyber v3's neural training schedule, model-specific refit cadence, four scored stations and 16 graph nodes. It retains Đức's independently implemented models, including the pinned official TimeXer adapter and a separately implemented SARIMA filter. Differences in architecture, implementation and runtime preclude byte-for-byte or numeric equivalence to PR #104. Configuration is recorded in the run directory. The tuning and peer-agreement checks in Issue #96 remain open.

Run the setup and model cells in order. The notebook uses the existing pilot archives in MyDrive and writes each session to a new directory under MyDrive/issue96-huuduc. The spatial cell needs the publisher archive UrbanEVDataset.zip. It tries the original download if the archive is absent.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import hashlib
import os
import subprocess
import sys
import zipfile
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")
SOURCE_ZIP = DRIVE / "urbanev-source.zip"
ZIP_PATH = DRIVE / "inputs-urbanev-SZH-v1.zip"
DEV_COMMIT = "dffd16a0b37df336330ef9b9d23056facd388681"
REPO = Path(f"/content/adaptive-retention-ev-source-{DEV_COMMIT[:12]}")
BRANCH = "feat/duc-urbanev-baseline-benchmark"
PROFILE = "data/inputs/issue18-input-handoff-v1-20261019/urbanev-SZH-14d"
INPUT_ROOT = REPO / PROFILE
ZIP_SHA = "2b8d1a47edebe066543f8f0aec5897c23945edcef62ba10357b1cb95e5ba27c7"
MANIFEST_SHA = "66c82dd38bd6e2c50af06bd1363605b033878289283997fe11f314bf8c2418d9"

# Git blob IDs of the repository modules imported by this notebook.
SOURCE_BLOBS = {
    "ev_baselines/__init__.py": "c984d9f330571bedb01e5060eec6fd66480b1a1f",
    "ev_baselines/naive.py": "1ed9cec112fa418e4bd35b96cb6ef97f7582243e",
    "ev_models/__init__.py": "5184ce76653d597f72b8d5adb2798b6c600cc82f",
    "ev_models/data.py": "d6f722283dfc3c6bc894d4f9bb363c8323934b75",
    "ev_models/model.py": "313b2fb3e307e224e4086bfe5f6f07b62f71bd1a",
    "ev_models/training.py": "770d43a92f1516021fe7e6148b2796a4aeb09a3e",
    "ev_models/checkpoint.py": "ecb0d1b7c1a9e0f4ebbfd14f2dc8a8113204193f",
    "ev_runner/__init__.py": "f7d610470a401af2746ffa637b97ab6eaaca95ca",
    "ev_runner/replay.py": "ff73f7010219d6dcc2e0881184621bddb429ec45",
    "ev_inputs/__init__.py": "f3be9997f117556586d483fe360dbb41a52a066a",
    "ev_inputs/features.py": "710339cf1c05d88bf205e65d3f1a7404d4708ee2",
    "ev_inputs/loader.py": "d55623f5bc39bc398c681467fce9fd8e0db021c1",
    "ev_inputs/splits.py": "675496da5bc081707ff793a45f17d5fe62d72483",
    "ev_inputs/windows.py": "8678cbaa768e26869fb150b1bdcc1cea44e6974d",
    "ev_inputs/hourly_reader.py": "172aeee04d68dda2b06a95fdb004f3c0ea30be72",
    "ev_inputs/scaling.py": "981c76faa31d25d06192fb5af3e0081f9181517d",
}


def sha(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def git(*args, cwd=REPO):
    return subprocess.check_output(["git", *args], cwd=cwd, text=True).strip()


def install_source(path, destination):
    prefix = f"adaptive-retention-ev-{DEV_COMMIT}/"
    with zipfile.ZipFile(path) as archive:
        members = archive.namelist()
        modules = {}
        for relative, expected in SOURCE_BLOBS.items():
            entry = prefix + relative
            if members.count(entry) != 1:
                raise ValueError(f"Expected {entry} once. Download the pinned source ZIP linked above.")
            data = archive.read(entry)
            header = f"blob {len(data)}\0".encode()
            if hashlib.sha1(header + data).hexdigest() != expected:
                raise ValueError(f"Source file does not match the pinned commit: {relative}")
            modules[relative] = data
    for relative, data in modules.items():
        target = destination / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)


if not SOURCE_ZIP.is_file():
    raise FileNotFoundError(f"Upload the GitHub source ZIP as {SOURCE_ZIP}")
if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"Upload the original dataset ZIP to {ZIP_PATH}")
if ZIP_PATH.stat().st_size != 151254390 or sha(ZIP_PATH) != ZIP_SHA:
    raise ValueError("UrbanEV ZIP size or SHA-256 mismatch")
install_source(SOURCE_ZIP, REPO)
SOURCE_ZIP_SHA = sha(SOURCE_ZIP)
if not INPUT_ROOT.exists():
    with zipfile.ZipFile(ZIP_PATH) as archive:
        names = [name for name in archive.namelist()
                 if name.startswith(PROFILE + "/") and not name.endswith("/")]
        expected = {"training.jsonl", "forecasts.jsonl", "labels.jsonl",
                    "entities.jsonl", "schedule.jsonl", "manifest.json"}
        if len(names) != len(expected) or set(names) != {f"{PROFILE}/{name}" for name in expected}:
            raise ValueError("Unexpected dataset archive members")
        archive.extractall(REPO, names)
if sha(INPUT_ROOT / "manifest.json") != MANIFEST_SHA:
    raise ValueError("Prepared input manifest SHA-256 mismatch")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Source module hashes PASS | commit:", DEV_COMMIT)
print("ZIP and manifest SHA-256 PASS")


Mounted at /content/drive
Source module hashes PASS | commit: dffd16a0b37df336330ef9b9d23056facd388681
ZIP and manifest SHA-256 PASS


In [2]:
%pip -q install scikit-learn==1.6.1 statsmodels==0.14.5 xgboost==3.0.5 reformer-pytorch==1.4.4 einops==0.8.1 gdown==5.2.0


In [3]:
import csv
import importlib.metadata
import json
import platform
import resource
import time
import traceback
import uuid
import warnings
from collections import defaultdict
from datetime import datetime, timedelta
from types import SimpleNamespace
from zoneinfo import ZoneInfo

import joblib
import numpy as np
import pandas as pd
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import torch
from torch import nn
from ev_baselines.naive import SeasonalNaive
from ev_models.data import InputRun
from ev_models.model import ForecastGRU, forecast_loss
from ev_models.training import seed_everything
from ev_runner.replay import Replay

BENCHMARK = True
PILOT_TARGET = "2152"
TARGETS = ("1405", "2152", "1153", "1693")
SEEDS = (42, 43, 44) if BENCHMARK else (42,)
TARGET_IDS = TARGETS if BENCHMARK else (PILOT_TARGET,)
SOURCE_EPOCHS = 3
SOURCE_BATCH_SIZE = 256
SOURCE_LR = 1e-3
TARGET_BATCH_SIZE = 4
TARGET_LR = 5e-4
GRAPH_EPOCHS = 5
GRAPH_LR = 1e-3
MODEL_NAMES = ("daily_naive", "seasonal_naive", "ridge", "sarima", "random_forest", "xgboost",
               "mlp", "gru", "lstm", "tcn", "timexer", "gcn_lstm")
REFIT_DAYS = 7
INITIAL_EPOCHS = 3
ONLINE_EPOCHS = 1
NEURAL_MODELS = MODEL_NAMES[6:]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DAY = timedelta(days=1)
HOUR = timedelta(hours=1)
TIME_XER_SHA = "76011909357972bd55a27adba2e1be994d81b327"

# Publisher station-distance matrix. A verified ZIP is downloaded only for the graph.
RAW_ZIP = DRIVE / "UrbanEVDataset.zip"
RAW_SHA = "9d3f0aec34434546d082509efcdeeaea116eaa841701cc5854a9b62a27881a79"
RAW_ID = "1OEpo-XDocd33aK3MbgDt9S5bcqwiAFPQ"
DISTANCE_MEMBER = "UrbanEVDataset/20220901-20230228_station-raw/station_distance.csv"
GRAPH_SUPPORTS = {
    "1405": ("1398", "1439", "1417"),
    "2152": ("2153", "2120", "2173"),
    "1153": ("1140", "1169", "1165"),
    "1693": ("1723", "1836", "1810"),
}
GRAPH_SUPPORT_IDS = tuple(t for supports in GRAPH_SUPPORTS.values() for t in supports)

phase = "benchmark" if BENCHMARK else "pilot"
stamp = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%Y%m%d-%H%M%S")
OUT = DRIVE / "issue96-huuduc" / f"{phase}-{stamp}-{uuid.uuid4().hex[:6]}"
OUT.mkdir(parents=True)
inputs = InputRun(INPUT_ROOT, MANIFEST_SHA)
for name in ("training.jsonl", "forecasts.jsonl", "labels.jsonl", "schedule.jsonl", "entities.jsonl"):
    inputs.verify(name)
if inputs.config["history_days"] != 14:
    raise ValueError("Expected 14 days of initial history")
counts = {t: inputs.entities[(*inputs.namespace, t)]["initial_trainable"] for t in TARGETS}
source_count = sum(row["initial_trainable"] for row in inputs.entities.values()
                   if row["role"] == "source")
print("Initial counts:", counts, "| Source count from metadata:", source_count)
if list(counts.values()) != [4, 12, 13, 13] or source_count != 65888:
    raise ValueError("Input population differs from the issue's historical counts")

def write_json(path, value):
    with path.open("x", encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)

def cpu_model():
    if Path("/proc/cpuinfo").is_file():
        for line in Path("/proc/cpuinfo").read_text().splitlines():
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    return platform.processor()

config = {
    "pilot_notebook_sha256": "976d270b385299d65dfa698a2555e93f94b2cb7e91d6685392e68f14899230e6",
    "phase": phase, "run_root": OUT.name, "git_commit": DEV_COMMIT, "branch": BRANCH,
    "zip_sha256": ZIP_SHA, "manifest_sha256": MANIFEST_SHA,
    "input_run": inputs.manifest["run_id"], "source_identity": inputs.source_hash,
    "targets": TARGET_IDS, "target_initial_counts": counts, "source_count": source_count,
    "days": 90, "seeds": SEEDS, "retention_lambda": 0,
    "source_epochs": SOURCE_EPOCHS, "initial_epochs": INITIAL_EPOCHS,
    "online_epochs": ONLINE_EPOCHS,
    "refits": {"ridge_days": 1, "sarima_days": REFIT_DAYS, "sarima_window_hours": 336,
               "tree_days": REFIT_DAYS},
    "source_batch_size": SOURCE_BATCH_SIZE, "source_lr": SOURCE_LR,
    "target_batch_size": TARGET_BATCH_SIZE, "target_lr": TARGET_LR,
    "graph_initial_epochs": GRAPH_EPOCHS, "graph_lr": GRAPH_LR,
    "source_checkpoint_selection": "fixed final epoch",
    "timexer_commit": TIME_XER_SHA,
    "graph_supports": GRAPH_SUPPORTS, "graph_distance_member": DISTANCE_MEMBER,
    "cyber_v3_reference": "0a0df67",
    "implementation": "Duc pilot lineage; official TimeXer and independent model code",
    "device": str(DEVICE), "cpu": cpu_model(), "cores": os.cpu_count(),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "cuda": torch.version.cuda, "python": platform.python_version(), "os": platform.platform(),
    "packages": {p: importlib.metadata.version(p) for p in
                 ("numpy", "pandas", "torch", "scikit-learn", "statsmodels", "xgboost",
                  "reformer-pytorch", "einops", "joblib")},
}
write_json(OUT / "session.json", config)
print("Environment:", DEVICE, config["gpu"] or config["cpu"])
print("Output:", OUT)


Initial counts: {'1405': 4, '2152': 12, '1153': 13, '1693': 13} | Source count from metadata: 65888
Environment: cpu Intel(R) Xeon(R) CPU @ 2.20GHz
Output: /content/drive/MyDrive/issue96-huuduc/benchmark-20261111-003045-e3e4e0


In [4]:
class DailyNaive(SeasonalNaive):
    def predict(self, record):
        self.ingest(record)
        origin = datetime.fromisoformat(record["origin"])
        past = [value for stamp, value in self.history.items()
                if datetime.fromisoformat(stamp) < origin]
        output = []
        for hour in range(24):
            key = (origin - DAY + hour * HOUR).isoformat()
            if key in self.history:
                output.append(self.history[key])
                self.fallbacks["daily"] += 1
            elif past:
                output.append(float(np.mean(past)))
                self.fallbacks["mean"] += 1
            else:
                output.append(0.0)
                self.fallbacks["zero"] += 1
        return output


def features(record):
    return (np.asarray(record["x_hist"], dtype=np.float32),
            np.asarray(record["c_future"], dtype=np.float32))


def flat(record):
    return np.concatenate([part.ravel() for part in features(record)])


class Tabular:
    def __init__(self, name, seed):
        from sklearn.ensemble import RandomForestRegressor
        from sklearn.linear_model import Ridge

        if name == "ridge":
            self.estimator = Ridge(alpha=1.0)
        elif name == "random_forest":
            self.estimator = RandomForestRegressor(
                n_estimators=100, max_depth=10, n_jobs=2, random_state=seed
            )
        elif name == "xgboost":
            from xgboost import XGBRegressor
            self.estimator = XGBRegressor(
                n_estimators=50, max_depth=5, learning_rate=0.1,
                tree_method="hist", n_jobs=2, random_state=seed,
            )
        else:
            raise ValueError(name)

    def fit(self, rows):
        x = np.stack([flat(row) for row in rows])
        y = np.asarray([row["y_scaled"] for row in rows])
        self.estimator.fit(x, y)

    def predict(self, record):
        return self.estimator.predict(flat(record)[None, :])[0]


class Sarima:
    ORDERS = (((1, 0, 1), (1, 0, 0, 24)),)

    def __init__(self, start):
        self.start = start
        self.history = {}
        self.order = None
        self.params = None
        self.fit_log = []
        self.state_start = start

    def add(self, record, include_history):
        if include_history:
            for stamp, row in zip(record["history_labels"], record["x_hist"], strict=True):
                if row[-1]:
                    self.history[datetime.fromisoformat(stamp)] = row[0]
        for stamp, value in zip(record["future_labels"], record["y_scaled"], strict=True):
            self.history[datetime.fromisoformat(stamp)] = value

    def model(self, start, stop, order):
        from statsmodels.tsa.statespace.sarimax import SARIMAX
        index = pd.date_range(start, stop - HOUR, freq="h")
        series = np.asarray([self.history.get(t, np.nan) for t in index], dtype=float)
        return SARIMAX(series, order=order[0], seasonal_order=order[1])

    def fit(self, stop):
        choices = [self.order] if self.order is not None else self.ORDERS
        candidates = []
        state_start = max(self.start, stop - 336 * HOUR)
        for order in choices:
            event = {"stop": stop.isoformat(), "order": order}
            try:
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always")
                    result = self.model(state_start, stop, order).fit(disp=False, maxiter=15)
                converged = bool(result.mle_retvals.get("converged", False))
                finite = np.isfinite(result.aic) and np.isfinite(result.params).all()
                event.update(aic=float(result.aic) if np.isfinite(result.aic) else None,
                             converged=converged, warnings=[str(w.message) for w in caught])
                if finite:
                    candidates.append((result.aic, order, result.params))
            except (ValueError, np.linalg.LinAlgError) as exc:
                event["failure"] = repr(exc)
            self.fit_log.append(event)
        if not candidates:
            if self.params is None:
                raise RuntimeError(f"No finite initial SARIMA fit: {self.fit_log}")
            return  # Keep previous parameters; log the failed refit above.
        _, self.order, self.params = min(candidates, key=lambda entry: entry[0])
        self.state_start = state_start

    def predict(self, record):
        origin = datetime.fromisoformat(record["origin"])
        # Update the state on arrived observations with the last fitted parameters.
        return self.model(self.state_start, origin, self.order).filter(self.params).forecast(24)


In [5]:
class CausalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation):
        super().__init__()
        self.left_pad = 2 * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, 3, dilation=dilation)
        self.skip = (nn.Identity() if in_channels == out_channels
                     else nn.Conv1d(in_channels, out_channels, 1))

    def forward(self, x):
        return torch.relu(self.conv(nn.functional.pad(x, (self.left_pad, 0))) + self.skip(x))


def head(width):
    return nn.Sequential(nn.Linear(width + 120, 64), nn.ReLU(),
                         nn.Linear(64, 24), nn.Softplus())


class LocalNetwork(nn.Module):
    def __init__(self, name):
        super().__init__()
        self.name = name
        if name == "mlp":
            self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(168, 128), nn.ReLU(),
                                         nn.Linear(128, 64), nn.ReLU())
        elif name == "lstm":
            self.encoder = nn.LSTM(7, 64, num_layers=2, batch_first=True)
        elif name == "tcn":
            dilations = (1, 2, 4, 8)
            self.encoder = nn.Sequential(*[CausalBlock(7 if i == 0 else 64, 64, d)
                                           for i, d in enumerate(dilations)])
        else:
            raise ValueError(name)
        self.head = head(64)

    def forward(self, x, calendar):
        if self.name == "lstm":
            history = self.encoder(x)[0][:, -1]
        elif self.name == "tcn":
            history = self.encoder(x.transpose(1, 2))[:, :, -1]
        else:
            history = self.encoder(x)
        return self.head(torch.cat([history, calendar.flatten(1)], dim=1))


class TimeXerAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        root = Path("/content/TimeXer")
        if not root.exists():
            git("clone", "https://github.com/thuml/TimeXer.git", str(root), cwd="/content")
            git("checkout", "--detach", TIME_XER_SHA, cwd=root)
        if git("rev-parse", "HEAD", cwd=root) != TIME_XER_SHA:
            raise ValueError("Unexpected TimeXer commit")
        sys.path.insert(0, str(root))
        from models.TimeXer import Model

        self.backbone = Model(SimpleNamespace(
            task_name="short_term_forecast", features="MS", seq_len=24, pred_len=24,
            patch_len=6, enc_in=7, use_norm=False, d_model=32, d_ff=64, n_heads=4,
            e_layers=1, dropout=0.0, embed="timeF", freq="h", factor=1, activation="gelu",
        ))

    def forward(self, x, calendar):
        # Official MS mode expects the target (scaled historical energy) in the last column.
        reordered = torch.cat([x[:, :, 1:], x[:, :, :1]], dim=2)
        output = self.backbone(reordered, calendar, None, None)
        return nn.functional.softplus(output.squeeze(-1))


class GraphLSTM(nn.Module):
    def __init__(self, adjacency):
        super().__init__()
        self.register_buffer("adjacency", torch.as_tensor(adjacency, dtype=torch.float32))
        self.gcn = nn.Linear(7, 32)
        self.lstm = nn.LSTM(32, 64, batch_first=True)
        self.head = head(64)

    def forward(self, x, calendar):
        # x is [batch, node, 24, 7].
        batch, nodes, _, _ = x.shape
        aggregated = torch.einsum("ij,bjtf->bitf", self.adjacency, x)
        temporal = torch.relu(self.gcn(aggregated)).flatten(0, 1)
        hidden = self.lstm(temporal)[0][:, -1].reshape(batch, nodes, 64)
        known = calendar.flatten(1)[:, None, :].expand(-1, nodes, -1)
        return self.head(torch.cat([hidden, known], dim=-1))


def network(name, adjacency=None):
    if name == "gru":
        return ForecastGRU()
    if name == "timexer":
        return TimeXerAdapter()
    if name == "gcn_lstm":
        return GraphLSTM(adjacency)
    return LocalNetwork(name)


def arrays(samples):
    x, c, y, nodes = zip(*samples, strict=True)
    return tuple(np.asarray(part, dtype=dtype) for part, dtype in
                 zip((x, c, y, nodes), (np.float32, np.float32, np.float32, np.int64),
                     strict=True))


def sample(record):
    return (*features(record), record["y_scaled"], 0)


def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


class Neural:
    def __init__(self, name, seed, adjacency=None, source_state=None, *, lr=TARGET_LR):
        seed_everything(seed)
        self.model = network(name, adjacency).to(DEVICE)
        if source_state is not None:
            self.model.load_state_dict(source_state)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)

    def fit(self, data, epochs, batch_size, *, shuffle=True):
        y = data[2]
        self.model.train()
        for _ in range(epochs):
            order = torch.randperm(len(y)) if shuffle else torch.arange(len(y))
            for idx in order.split(batch_size):
                x_batch, c_batch, truth, node = (
                    torch.as_tensor(part[idx.numpy()], device=DEVICE) for part in data
                )
                self.optimizer.zero_grad(set_to_none=True)
                output = self.model(x_batch, c_batch)
                if output.ndim == 3:
                    output = output[torch.arange(len(idx), device=DEVICE), node]
                loss = forecast_loss(output, truth)
                loss.backward()
                self.optimizer.step()

    @torch.inference_mode()
    def predict(self, data):
        self.model.eval()
        x, c = [torch.as_tensor(part[None], device=DEVICE) for part in data]
        return self.model(x, c)[0].cpu().numpy()


In [6]:
from functools import lru_cache

torch.set_num_threads(min(4, os.cpu_count() or 1))

@lru_cache(maxsize=None)
def target_data(station):
    rows = inputs.training("development", station)
    replay = Replay(inputs, station, days=90)
    replay.prepare_label_index()
    return rows, replay


@lru_cache(maxsize=1)
def source_data():
    rows = inputs.training("source")
    if len(rows) != source_count:
        raise ValueError("Decoded Source count differs from verified metadata")
    return arrays([sample(record) for record in rows])


@lru_cache(maxsize=None)
def train_source(name, seed):
    data = source_data()
    path = OUT / f"source-{name}-{seed}-{uuid.uuid4().hex[:6]}"
    path.mkdir()
    neural = Neural(name, seed, lr=SOURCE_LR)
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    sync()
    elapsed = 0.0
    for epoch in range(1, SOURCE_EPOCHS + 1):
        started = time.perf_counter()
        neural.fit(data, 1, SOURCE_BATCH_SIZE)
        sync()
        elapsed += time.perf_counter() - started
        if epoch == SOURCE_EPOCHS:
            destination = path / f"epoch-{epoch:02d}.pt"
            torch.save(neural.model.state_dict(), destination)
    metadata = {"model": name, "seed": seed, "samples": len(data[2]),
                "source_train_seconds": elapsed, "checkpoint": str(destination),
                "sha256": sha(destination), "peak_cuda_allocated_bytes":
                torch.cuda.max_memory_allocated() if DEVICE.type == "cuda" else None}
    write_json(path / "source.json", metadata)
    return metadata


@lru_cache(maxsize=1)
def graph_config():
    raw_path = RAW_ZIP
    if not raw_path.is_file():
        import gdown
        raw_path = Path("/content/UrbanEVDataset.zip")
        if not raw_path.is_file():
            gdown.download(id=RAW_ID, output=str(raw_path), quiet=False)
    if not raw_path.is_file() or raw_path.stat().st_size != 647305188 or sha(raw_path) != RAW_SHA:
        raise ValueError("Publisher UrbanEV ZIP is missing or fails the recorded size/SHA-256 check")
    ids = (*TARGETS, *GRAPH_SUPPORT_IDS)
    if len(set(ids)) != 16:
        raise ValueError("Expected sixteen distinct graph stations")
    for station in ids:
        inputs.check_target(station)
    with zipfile.ZipFile(raw_path) as archive:
        with archive.open(DISTANCE_MEMBER) as stream:
            distance_frame = pd.read_csv(stream, index_col=0)
    distance_frame.index = distance_frame.index.map(str)
    distance_frame.columns = distance_frame.columns.map(str)
    distances = distance_frame.loc[list(ids), list(ids)].to_numpy(dtype=float)
    if not np.isfinite(distances).all() or not np.allclose(distances, distances.T):
        raise ValueError("Station distance matrix must be finite and symmetric")
    sigma = float(np.std(distances[distances > 0]))
    if sigma <= 0:
        raise ValueError("Invalid geographic distance scale")
    links = np.exp(-np.square(distances / sigma))
    np.fill_diagonal(links, 1.0)
    degree = links.sum(axis=1) ** -0.5
    return ids, (links * degree[:, None] * degree[None, :]).astype(np.float32), sigma


def graph_features(records, ids, scored_station):
    _, calendar = features(records[scored_station])
    x = np.stack([features(records[t])[0] if records[t] is not None
                  else np.zeros((24, 7), dtype=np.float32) for t in ids])
    return x, calendar


def graph_initial(initial, ids, station):
    rows = {t: {r["origin"]: r for r in initial[t]} for t in ids}
    samples = []
    for origin, record in sorted(rows[station].items()):
        records = {t: rows[t].get(origin) for t in ids}
        x, c = graph_features(records, ids, station)
        samples.append((x, c, record["y_scaled"], ids.index(station)))
    return arrays(samples), len(samples)


def add_event(path, event):
    with path.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(event, ensure_ascii=False, allow_nan=False) + "\n")


def fit_and_replay(name, mode, scored, seed, folder, totals):
    graph = name == "gcn_lstm"
    neural_model = name in NEURAL_MODELS
    single = scored[0]
    ids, adjacency, sigma = graph_config() if graph else ((single,), None, None)
    if graph:
        write_json(folder / "graph.json", {"nodes": ids, "scored": scored,
                   "adjacency": adjacency.tolist(), "sigma": sigma,
                   "distance_member": DISTANCE_MEMBER, "raw_zip_sha256": RAW_SHA,
                   "support_lookup": "same-origin predictor histories only"})
    initial, replays = {}, {}
    for station in ids:
        initial[station], replays[station] = target_data(station)
    reference = replays[single]
    start, stop = reference.start, reference.stop
    history = list(initial[single])
    source = train_source(name, seed) if mode == "transfer" else None
    totals["source_train_seconds"] = source["source_train_seconds"] if source else None
    totals["source_checkpoint_sha256"] = source["sha256"] if source else None

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    sync()
    started = time.perf_counter()
    if neural_model:
        state = (torch.load(source["checkpoint"], map_location="cpu", weights_only=True)
                 if source else None)
        predictor = Neural(name, seed, adjacency, source_state=state,
                           lr=GRAPH_LR if graph else TARGET_LR)
        if graph:
            data, totals["graph_initial_days"] = graph_initial(initial, ids, single)
            predictor.fit(data, GRAPH_EPOCHS, 1, shuffle=False)
        else:
            data = arrays([sample(record) for record in history])
            predictor.fit(data, INITIAL_EPOCHS, TARGET_BATCH_SIZE)
        totals["parameters"] = sum(p.numel() for p in predictor.model.parameters())
    elif name in {"daily_naive", "seasonal_naive"}:
        predictor = (DailyNaive if name == "daily_naive" else SeasonalNaive)(history)
    elif name == "sarima":
        predictor = Sarima(start - 14 * DAY)
        for record in history:
            predictor.add(record, include_history=True)
        predictor.fit(start)
    else:
        predictor = Tabular(name, seed)
        predictor.fit(history)
    sync()
    totals["initial_seconds"] = time.perf_counter() - started
    totals.update(inference_seconds=0.0, update_refit_seconds=0.0)
    totals["target_loop_started"] = time.perf_counter()
    dirty = False
    last_fit = start
    refit_days = 1 if name == "ridge" else REFIT_DAYS
    pending = []
    path = folder / "daily.jsonl"
    for as_of in [start + day * DAY for day in range(91)]:
        next_pending = []
        for origin, station, pred, record, graph_data in pending:
            replay = replays[station]
            labels = replay.labels(origin, as_of)
            if labels.status == "pending":
                next_pending.append((origin, station, pred, record, graph_data))
                continue
            event = {"event": "label", "station": station, "origin": origin.isoformat(),
                     "as_of": as_of.isoformat(), "status": labels.status,
                     "available_at": labels.available_at.isoformat() if labels.available_at else None}
            if labels.values is None:
                add_event(path, event)
                continue
            y = np.asarray(labels.values, dtype=float)
            errors = pred - y
            mae, mse = float(np.abs(errors).mean()), float(np.square(errors).mean())
            add_event(path, {**event, "mae_kwh": mae, "mse_kwh2": mse,
                             "label_kwh": y.tolist()})
            totals["errors"][station].append((mae, mse))
            sync()
            updated = time.perf_counter()
            record = {**record, "y_scaled": (y / replay.scale).tolist()}
            if neural_model:
                row = (*graph_data, record["y_scaled"], ids.index(station)) if graph else sample(record)
                predictor.fit(arrays([row]), ONLINE_EPOCHS, 1)
            elif name in {"daily_naive", "seasonal_naive"}:
                predictor.update(record)
            elif name == "sarima":
                predictor.add(record, include_history=False)
                dirty = True
            else:
                history.append(record)
                dirty = True
            sync()
            totals["update_refit_seconds"] += time.perf_counter() - updated
        pending = next_pending

        if as_of >= stop:
            break
        if dirty and as_of - last_fit >= refit_days * DAY:
            updated = time.perf_counter()
            if name == "sarima":
                predictor.fit(as_of)
            else:
                predictor.fit(history)
            totals["update_refit_seconds"] += time.perf_counter() - updated
            last_fit, dirty = as_of, False

        records = {t: replays[t].forecast(as_of) for t in ids}
        issued = [t for t in scored if records[t] is not None]
        for station in scored:
            if station not in issued:
                add_event(path, {"event": "skip", "station": station,
                                 "origin": as_of.isoformat(),
                                 "status": replays[station].schedule[as_of]})
        if not issued:
            continue

        graph_data = graph_features(records, ids, single) if graph else None
        sync()
        predicted = time.perf_counter()
        if graph:
            predictions = predictor.predict(graph_data)
        elif neural_model:
            predictions = predictor.predict(features(records[single]))[None, :]
        else:
            predictions = np.asarray(predictor.predict(records[single]))[None, :]
        sync()
        totals["inference_seconds"] += time.perf_counter() - predicted

        for station in issued:
            p = np.asarray(predictions[ids.index(station)], dtype=float)
            if p.shape != (24,) or not np.isfinite(p).all():
                raise ValueError(f"Invalid prediction for {station} at {as_of}")
            p = np.maximum(p, 0.0) * replays[station].scale
            add_event(path, {"event": "forecast", "station": station,
                             "origin": as_of.isoformat(), "prediction_kwh": p.tolist()})
            totals["issued"][station] += 1
            pending.append((as_of, station, p, records[station], graph_data))
    totals["replay_seconds"] = time.perf_counter() - totals.pop("target_loop_started")
    for origin, station, *_ in pending:
        add_event(path, {"event": "label", "origin": origin.isoformat(),
                         "station": station, "status": "pending_at_stop"})
    if neural_model:
        torch.save({"model": predictor.model.state_dict(), "optimizer": predictor.optimizer.state_dict()},
                   folder / "estimator.pt")
    elif name in {"daily_naive", "seasonal_naive"}:
        write_json(folder / "estimator.json", predictor.snapshot())
    else:
        joblib.dump(predictor, folder / "estimator.joblib")
        if name == "sarima":
            write_json(folder / "sarima_fit_log.json", predictor.fit_log)


def run_job(name, mode, scored, seed):
    tag = f"{name}-{mode}-{'graph' if name == 'gcn_lstm' else scored[0]}-{seed}"
    run_id = f"{tag}-{uuid.uuid4().hex[:6]}"
    folder = OUT / run_id
    folder.mkdir()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    totals = {"errors": defaultdict(list), "issued": defaultdict(int)}
    failure = None
    try:
        fit_and_replay(name, mode, scored, seed, folder, totals)
    except Exception as error:
        failure = f"{type(error).__name__}: {error}"
        (folder / "failure.txt").write_text(traceback.format_exc(), encoding="utf-8")
        print(f"FAILED {run_id}: {failure}", flush=True)
    duration = time.perf_counter() - started
    errors, issued = totals.pop("errors"), totals.pop("issued")
    totals.pop("target_loop_started", None)
    totals["job_wall_seconds"] = duration
    totals["process_peak_rss_kib_since_kernel_start"] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    totals["cuda_peak_allocated_bytes"] = (
        torch.cuda.max_memory_allocated() if DEVICE.type == "cuda" else None)
    rows = []
    for station in scored:
        daily = errors[station]
        rows.append({"run_id": run_id, "model": name, "mode": mode, "target": station,
                     "seed": seed, "scheduled_days": 90, "issued_days": issued[station],
                     "scored_days": len(daily), "status": "failed" if failure else ("ok" if daily else "null"),
                     "mae_kwh": float(np.mean([a for a, _ in daily])) if daily and not failure else None,
                     "rmse_kwh": float(np.sqrt(np.mean([b for _, b in daily]))) if daily and not failure else None,
                     "error": failure, **totals})
    write_json(folder / "result.json", rows)
    write_json(folder / "sha256.json", {p.name: sha(p) for p in sorted(folder.iterdir()) if p.is_file()})
    return rows


def collect():
    rows = []
    for path in sorted(OUT.glob("*/result.json")):
        rows.extend(json.loads(path.read_text(encoding="utf-8")))
    data = pd.DataFrame(rows)
    if not data.empty:
        graph = data["model"] == "gcn_lstm"
        data.loc[~graph].to_csv(OUT / "runs.csv", index=False)
        data.loc[graph].to_csv(OUT / "runs_graph.csv", index=False)
    return data


def jobs_for(name):
    if name == "gcn_lstm":
        return [("spatial", (t,), seed) for seed in SEEDS for t in TARGET_IDS]
    modes = ("target_only", "transfer") if name in NEURAL_MODELS else ("target_only",)
    seeds = SEEDS if name in (*NEURAL_MODELS, "random_forest", "xgboost") else (None,)
    return [(mode, (t,), seed) for mode in modes for seed in seeds for t in TARGET_IDS]


def run_model(name):
    previous = collect()
    if not previous.empty and name in set(previous["model"]):
        raise ValueError("This model already has rows in this session. Use a new session for retries.")
    if name == "gcn_lstm":
        graph_config()
    jobs = jobs_for(name)
    for mode, stations, seed in jobs:
        print(name, mode, stations, seed, flush=True)
        run_job(name, mode, stations, seed)
        collect()
    frame = collect()
    columns = ["model", "mode", "target", "seed", "scheduled_days", "issued_days",
               "scored_days", "mae_kwh", "rmse_kwh", "status", "error"]
    display(frame.loc[frame["model"] == name, columns])


In [7]:
rows, replay = target_data(PILOT_TARGET)
assert len(rows) == counts[PILOT_TARGET]
assert len(replay.schedule) == 90
for origin in sorted(replay.schedule)[:4]:
    record = replay.forecast(origin)
    if record is not None:
        x, c = features(record)
        assert x.shape == (24, 7) and c.shape == (24, 5)
        assert replay.labels(origin, origin).status == "pending"
        print(origin.isoformat(), x.shape, c.shape, replay.schedule[origin])
print("Input shapes and availability gate PASS")

expected = {name: len(jobs_for(name)) for name in MODEL_NAMES}
assert sum(expected.values()) == 172
assert expected["gcn_lstm"] == 12
print("Execution matrix:", expected)


2022-12-01T00:00:00+08:00 (24, 7) (24, 5) created
2022-12-02T00:00:00+08:00 (24, 7) (24, 5) created
2022-12-03T00:00:00+08:00 (24, 7) (24, 5) created
2022-12-04T00:00:00+08:00 (24, 7) (24, 5) created
Input shapes and availability gate PASS
Execution matrix: {'daily_naive': 4, 'seasonal_naive': 4, 'ridge': 4, 'sarima': 4, 'random_forest': 12, 'xgboost': 12, 'mlp': 24, 'gru': 24, 'lstm': 24, 'tcn': 24, 'timexer': 24, 'gcn_lstm': 12}


In [8]:
run_model("daily_naive")


daily_naive target_only ('1405',) None
daily_naive target_only ('2152',) None
daily_naive target_only ('1153',) None
daily_naive target_only ('1693',) None


,model,mode,target,seed,scheduled_days,issued_days,scored_days,mae_kwh,rmse_kwh,status,error
0,daily_naive,target_only,1153,None,90,59,52,11.194111,14.376285,ok,None
1,daily_naive,target_only,1405,None,90,90,89,6.952293,9.413904,ok,None
2,daily_naive,target_only,1693,None,90,59,52,2.908253,4.760721,ok,None
3,daily_naive,target_only,2152,None,90,59,52,7.248197,9.730497,ok,None


In [9]:
run_model("seasonal_naive")


seasonal_naive target_only ('1405',) None
seasonal_naive target_only ('2152',) None
seasonal_naive target_only ('1153',) None
seasonal_naive target_only ('1693',) None


,model,mode,target,seed,scheduled_days,issued_days,scored_days,mae_kwh,rmse_kwh,status,error
4,seasonal_naive,target_only,1153,None,90,59,52,12.579060,16.338291,ok,None
5,seasonal_naive,target_only,1405,None,90,90,89,8.547090,11.296096,ok,None
6,seasonal_naive,target_only,1693,None,90,59,52,4.580662,6.482195,ok,None
7,seasonal_naive,target_only,2152,None,90,59,52,8.145633,10.466618,ok,None


In [10]:
run_model("ridge")


ridge target_only ('1405',) None
ridge target_only ('2152',) None
ridge target_only ('1153',) None
ridge target_only ('1693',) None


,model,mode,target,seed,scheduled_days,issued_days,scored_days,mae_kwh,rmse_kwh,status,error
4,ridge,target_only,1153,None,90,59,52,8.015456,10.309107,ok,None
5,ridge,target_only,1405,None,90,90,89,6.495201,8.842677,ok,None
6,ridge,target_only,1693,None,90,59,52,2.827512,4.176927,ok,None
7,ridge,target_only,2152,None,90,59,52,6.397517,8.356665,ok,None


In [11]:
run_model("sarima")


sarima target_only ('1405',) None
sarima target_only ('2152',) None
sarima target_only ('1153',) None
sarima target_only ('1693',) None


,model,mode,target,seed,scheduled_days,issued_days,scored_days,mae_kwh,rmse_kwh,status,error
8,sarima,target_only,1153,None,90,59,52,10.216997,13.817093,ok,None
9,sarima,target_only,1405,None,90,90,89,7.535180,10.689447,ok,None
10,sarima,target_only,1693,None,90,59,52,2.753515,4.526243,ok,None
11,sarima,target_only,2152,None,90,59,52,6.202609,7.779306,ok,None


In [12]:
run_model("random_forest")


random_forest target_only ('1405',) 42
random_forest target_only ('2152',) 42
random_forest target_only ('1153',) 42
random_forest target_only ('1693',) 42
random_forest target_only ('1405',) 43
random_forest target_only ('2152',) 43
random_forest target_only ('1153',) 43
random_forest target_only ('1693',) 43
random_forest target_only ('1405',) 44
random_forest target_only ('2152',) 44
random_forest target_only ('1153',) 44
random_forest target_only ('1693',) 44


,model,mode,target,seed,scheduled_days,issued_days,scored_days,mae_kwh,rmse_kwh,status,error
4,random_forest,target_only,1153,42.0,90,59,52,7.727591,10.131667,ok,None
5,random_forest,target_only,1153,43.0,90,59,52,7.657007,10.036526,ok,None
6,random_forest,target_only,1153,44.0,90,59,52,7.741686,10.143199,ok,None
7,random_forest,target_only,1405,42.0,90,90,89,6.647297,8.911259,ok,None
8,random_forest,target_only,1405,43.0,90,90,89,6.652684,8.923032,ok,None
9,random_forest,target_only,1405,44.0,90,90,89,6.537165,8.844237,ok,None
10,random_forest,target_only,1693,42.0,90,59,52,2.890910,4.284499,ok,None
11,random_forest,target_only,1693,43.0,90,59,52,2.859655,4.230428,ok,None
12,random_forest,target_only,1693,44.0,90,59,52,2.852453,4.249103,ok,None
13,random_forest,target_only,2152,42.0,90,59,52,6.059617,7.474077,ok,None


In [13]:
run_model("xgboost")


xgboost target_only ('1405',) 42
xgboost target_only ('2152',) 42
xgboost target_only ('1153',) 42
xgboost target_only ('1693',) 42
xgboost target_only ('1405',) 43
xgboost target_only ('2152',) 43
xgboost target_only ('1153',) 43
xgboost target_only ('1693',) 43
xgboost target_only ('1405',) 44
xgboost target_only ('2152',) 44
xgboost target_only ('1153',) 44
xgboost target_only ('1693',) 44


,model,mode,target,seed,scheduled_days,issued_days,scored_days,mae_kwh,rmse_kwh,status,error
28,xgboost,target_only,1153,42.0,90,59,52,8.214931,11.048253,ok,None
29,xgboost,target_only,1153,43.0,90,59,52,8.214931,11.048253,ok,None
30,xgboost,target_only,1153,44.0,90,59,52,8.214931,11.048253,ok,None
31,xgboost,target_only,1405,42.0,90,90,89,7.463919,10.312227,ok,None
32,xgboost,target_only,1405,43.0,90,90,89,7.463919,10.312227,ok,None
33,xgboost,target_only,1405,44.0,90,90,89,7.463919,10.312227,ok,None
34,xgboost,target_only,1693,42.0,90,59,52,3.098231,4.723938,ok,None
35,xgboost,target_only,1693,43.0,90,59,52,3.098231,4.723938,ok,None
36,xgboost,target_only,1693,44.0,90,59,52,3.098231,4.723938,ok,None
37,xgboost,target_only,2152,42.0,90,59,52,7.068171,9.119463,ok,None


In [14]:
run_model("mlp")


mlp target_only ('1405',) 42
mlp target_only ('2152',) 42
mlp target_only ('1153',) 42
mlp target_only ('1693',) 42
mlp target_only ('1405',) 43
mlp target_only ('2152',) 43
mlp target_only ('1153',) 43
mlp target_only ('1693',) 43
mlp target_only ('1405',) 44
mlp target_only ('2152',) 44
mlp target_only ('1153',) 44
mlp target_only ('1693',) 44
mlp transfer ('1405',) 42
mlp transfer ('2152',) 42
mlp transfer ('1153',) 42
mlp transfer ('1693',) 42
mlp transfer ('1405',) 43
mlp transfer ('2152',) 43
mlp transfer ('1153',) 43
mlp transfer ('1693',) 43
mlp transfer ('1405',) 44
mlp transfer ('2152',) 44
mlp transfer ('1153',) 44
mlp transfer ('1693',) 44


,model,mode,target,seed,scheduled_days,issued_days,scored_days,mae_kwh,rmse_kwh,status,error
4,mlp,target_only,1153,42.0,90,59,52,10.833329,13.419404,ok,None
5,mlp,target_only,1153,43.0,90,59,52,10.786615,13.253147,ok,None
6,mlp,target_only,1153,44.0,90,59,52,10.795659,13.411996,ok,None
7,mlp,target_only,1405,42.0,90,90,89,9.251526,11.750117,ok,None
8,mlp,target_only,1405,43.0,90,90,89,9.028058,11.516954,ok,None
9,mlp,target_only,1405,44.0,90,90,89,9.201426,11.751372,ok,None
10,mlp,target_only,1693,42.0,90,59,52,4.847919,5.746227,ok,None
11,mlp,target_only,1693,43.0,90,59,52,4.888173,5.805981,ok,None
12,mlp,target_only,1693,44.0,90,59,52,4.953435,5.874199,ok,None
13,mlp,target_only,2152,42.0,90,59,52,9.837569,12.295273,ok,None


In [15]:
run_model("gru")


gru target_only ('1405',) 42
gru target_only ('2152',) 42
gru target_only ('1153',) 42
gru target_only ('1693',) 42
gru target_only ('1405',) 43
gru target_only ('2152',) 43
gru target_only ('1153',) 43
gru target_only ('1693',) 43
gru target_only ('1405',) 44
gru target_only ('2152',) 44
gru target_only ('1153',) 44
gru target_only ('1693',) 44
gru transfer ('1405',) 42
gru transfer ('2152',) 42
gru transfer ('1153',) 42
gru transfer ('1693',) 42
gru transfer ('1405',) 43
gru transfer ('2152',) 43
gru transfer ('1153',) 43
gru transfer ('1693',) 43
gru transfer ('1405',) 44
gru transfer ('2152',) 44
gru transfer ('1153',) 44
gru transfer ('1693',) 44


,model,mode,target,seed,scheduled_days,issued_days,scored_days,mae_kwh,rmse_kwh,status,error
4,gru,target_only,1153,42.0,90,59,52,11.370270,13.939647,ok,None
5,gru,target_only,1153,43.0,90,59,52,10.771572,13.350525,ok,None
6,gru,target_only,1153,44.0,90,59,52,11.536703,14.070154,ok,None
7,gru,target_only,1405,42.0,90,90,89,9.190266,11.713519,ok,None
8,gru,target_only,1405,43.0,90,90,89,8.816514,11.266516,ok,None
9,gru,target_only,1405,44.0,90,90,89,9.543968,12.079818,ok,None
10,gru,target_only,1693,42.0,90,59,52,4.960822,5.815304,ok,None
11,gru,target_only,1693,43.0,90,59,52,4.739445,5.616915,ok,None
12,gru,target_only,1693,44.0,90,59,52,5.114556,5.934096,ok,None
13,gru,target_only,2152,42.0,90,59,52,10.480403,12.817920,ok,None


In [16]:
run_model("lstm")


lstm target_only ('1405',) 42
lstm target_only ('2152',) 42
lstm target_only ('1153',) 42
lstm target_only ('1693',) 42
lstm target_only ('1405',) 43
lstm target_only ('2152',) 43
lstm target_only ('1153',) 43
lstm target_only ('1693',) 43
lstm target_only ('1405',) 44
lstm target_only ('2152',) 44
lstm target_only ('1153',) 44
lstm target_only ('1693',) 44
lstm transfer ('1405',) 42
lstm transfer ('2152',) 42
lstm transfer ('1153',) 42
lstm transfer ('1693',) 42
lstm transfer ('1405',) 43
lstm transfer ('2152',) 43
lstm transfer ('1153',) 43
lstm transfer ('1693',) 43
lstm transfer ('1405',) 44
lstm transfer ('2152',) 44
lstm transfer ('1153',) 44
lstm transfer ('1693',) 44


,model,mode,target,seed,scheduled_days,issued_days,scored_days,mae_kwh,rmse_kwh,status,error
28,lstm,target_only,1153,42.0,90,59,52,10.825246,13.333581,ok,None
29,lstm,target_only,1153,43.0,90,59,52,10.943500,13.625776,ok,None
30,lstm,target_only,1153,44.0,90,59,52,11.025132,13.454031,ok,None
31,lstm,target_only,1405,42.0,90,90,89,9.703472,12.198111,ok,None
32,lstm,target_only,1405,43.0,90,90,89,9.616331,12.194627,ok,None
33,lstm,target_only,1405,44.0,90,90,89,9.237639,11.686422,ok,None
34,lstm,target_only,1693,42.0,90,59,52,4.895812,5.786298,ok,None
35,lstm,target_only,1693,43.0,90,59,52,4.916262,5.797615,ok,None
36,lstm,target_only,1693,44.0,90,59,52,4.854824,5.719642,ok,None
37,lstm,target_only,2152,42.0,90,59,52,10.729725,12.994484,ok,None


In [17]:
run_model("tcn")


tcn target_only ('1405',) 42
tcn target_only ('2152',) 42
tcn target_only ('1153',) 42
tcn target_only ('1693',) 42
tcn target_only ('1405',) 43
tcn target_only ('2152',) 43
tcn target_only ('1153',) 43
tcn target_only ('1693',) 43
tcn target_only ('1405',) 44
tcn target_only ('2152',) 44
tcn target_only ('1153',) 44
tcn target_only ('1693',) 44
tcn transfer ('1405',) 42
tcn transfer ('2152',) 42
tcn transfer ('1153',) 42
tcn transfer ('1693',) 42
tcn transfer ('1405',) 43
tcn transfer ('2152',) 43
tcn transfer ('1153',) 43
tcn transfer ('1693',) 43
tcn transfer ('1405',) 44
tcn transfer ('2152',) 44
tcn transfer ('1153',) 44
tcn transfer ('1693',) 44


,model,mode,target,seed,scheduled_days,issued_days,scored_days,mae_kwh,rmse_kwh,status,error
100,tcn,target_only,1153,42.0,90,59,52,10.548677,13.071978,ok,None
101,tcn,target_only,1153,43.0,90,59,52,10.380303,12.923923,ok,None
102,tcn,target_only,1153,44.0,90,59,52,10.600242,13.087900,ok,None
103,tcn,target_only,1405,42.0,90,90,89,8.368000,10.799012,ok,None
104,tcn,target_only,1405,43.0,90,90,89,8.236595,10.472971,ok,None
105,tcn,target_only,1405,44.0,90,90,89,7.978132,10.215225,ok,None
106,tcn,target_only,1693,42.0,90,59,52,4.606019,5.485186,ok,None
107,tcn,target_only,1693,43.0,90,59,52,4.485236,5.401063,ok,None
108,tcn,target_only,1693,44.0,90,59,52,4.361992,5.314079,ok,None
109,tcn,target_only,2152,42.0,90,59,52,8.711705,11.216589,ok,None


In [18]:
run_model("timexer")


timexer target_only ('1405',) 42
timexer target_only ('2152',) 42
timexer target_only ('1153',) 42
timexer target_only ('1693',) 42
timexer target_only ('1405',) 43
timexer target_only ('2152',) 43
timexer target_only ('1153',) 43
timexer target_only ('1693',) 43
timexer target_only ('1405',) 44
timexer target_only ('2152',) 44
timexer target_only ('1153',) 44
timexer target_only ('1693',) 44
timexer transfer ('1405',) 42
timexer transfer ('2152',) 42
timexer transfer ('1153',) 42
timexer transfer ('1693',) 42
timexer transfer ('1405',) 43
timexer transfer ('2152',) 43
timexer transfer ('1153',) 43
timexer transfer ('1693',) 43
timexer transfer ('1405',) 44
timexer transfer ('2152',) 44
timexer transfer ('1153',) 44
timexer transfer ('1693',) 44


,model,mode,target,seed,scheduled_days,issued_days,scored_days,mae_kwh,rmse_kwh,status,error
124,timexer,target_only,1153,42.0,90,59,52,10.456163,13.041994,ok,None
125,timexer,target_only,1153,43.0,90,59,52,10.497512,13.110757,ok,None
126,timexer,target_only,1153,44.0,90,59,52,10.263042,12.741692,ok,None
127,timexer,target_only,1405,42.0,90,90,89,6.874280,8.920339,ok,None
128,timexer,target_only,1405,43.0,90,90,89,7.037198,9.318066,ok,None
129,timexer,target_only,1405,44.0,90,90,89,7.048166,9.479724,ok,None
130,timexer,target_only,1693,42.0,90,59,52,3.800414,4.898708,ok,None
131,timexer,target_only,1693,43.0,90,59,52,3.885542,4.914000,ok,None
132,timexer,target_only,1693,44.0,90,59,52,3.952491,4.989455,ok,None
133,timexer,target_only,2152,42.0,90,59,52,6.677803,8.598784,ok,None


In [19]:
run_model("gcn_lstm")


Downloading...
From (original): https://drive.google.com/uc?id=1OEpo-XDocd33aK3MbgDt9S5bcqwiAFPQ
From (redirected): https://drive.google.com/uc?id=1OEpo-XDocd33aK3MbgDt9S5bcqwiAFPQ&confirm=t&uuid=9f1edd62-d85b-4b82-a9d7-43b64f7d5bc3
To: /content/UrbanEVDataset.zip
100%|██████████| 647M/647M [00:09<00:00, 69.2MB/s]


gcn_lstm spatial ('1405',) 42
gcn_lstm spatial ('2152',) 42
gcn_lstm spatial ('1153',) 42
gcn_lstm spatial ('1693',) 42
gcn_lstm spatial ('1405',) 43
gcn_lstm spatial ('2152',) 43
gcn_lstm spatial ('1153',) 43
gcn_lstm spatial ('1693',) 43
gcn_lstm spatial ('1405',) 44
gcn_lstm spatial ('2152',) 44
gcn_lstm spatial ('1153',) 44
gcn_lstm spatial ('1693',) 44


,model,mode,target,seed,scheduled_days,issued_days,scored_days,mae_kwh,rmse_kwh,status,error
4,gcn_lstm,spatial,2152,42.0,90,59,52,7.194412,9.469811,ok,None
5,gcn_lstm,spatial,1693,42.0,90,59,52,3.408841,5.247441,ok,None
6,gcn_lstm,spatial,1153,42.0,90,59,52,9.761834,12.240417,ok,None
7,gcn_lstm,spatial,1405,42.0,90,90,89,8.274206,10.660556,ok,None
8,gcn_lstm,spatial,2152,43.0,90,59,52,7.301291,9.495851,ok,None
9,gcn_lstm,spatial,1693,43.0,90,59,52,3.406397,5.223755,ok,None
10,gcn_lstm,spatial,1153,43.0,90,59,52,9.720932,12.278723,ok,None
11,gcn_lstm,spatial,1405,43.0,90,90,89,8.450999,10.978179,ok,None
12,gcn_lstm,spatial,1693,44.0,90,59,52,3.511446,5.329858,ok,None
13,gcn_lstm,spatial,1153,44.0,90,59,52,9.839669,12.328932,ok,None


In [20]:
frame = collect()
completed = frame.loc[frame["status"] == "ok"]
coverage = {"1405": (90, 90, 89), "2152": (90, 59, 52),
            "1153": (90, 59, 52), "1693": (90, 59, 52)}
for row in completed.itertuples():
    assert (row.scheduled_days, row.issued_days, row.scored_days) == coverage[row.target], row
print(f"{len(completed)}/172 successful rows | output {OUT}")
display(frame.groupby(["model", "status"], dropna=False).size().rename("rows"))
if len(completed) != 172:
    raise RuntimeError("Benchmark incomplete. Inspect the failed or missing rows in the Drive session.")


172/172 successful rows | output /content/drive/MyDrive/issue96-huuduc/benchmark-20261111-003045-e3e4e0


,,rows
model,status,
daily_naive,ok,4
gcn_lstm,ok,12
gru,ok,24
lstm,ok,24
mlp,ok,24
random_forest,ok,12
ridge,ok,4
sarima,ok,4
seasonal_naive,ok,4
